In [26]:
# Importación de librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, cross_validate, ShuffleSplit
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.svm import SVR
from sklearn.model_selection import GridSearchCV

El primer paso será cargar el dataset desde el fichero csv.

In [27]:
df = pd.read_csv("acuario.csv", sep=",") #obtenemos el dataset desde el csv
df

,Especie,Peso,Long_vert,Long_diag,Long_tras,Altura,Ancho
0,0,242.0,23.2,25.4,30.0,11.5200,4.0200
1,0,290.0,24.0,26.3,31.2,12.4800,4.3056
2,0,340.0,23.9,26.5,31.1,12.3778,4.6961
3,0,363.0,26.3,29.0,33.5,12.7300,4.4555
4,0,430.0,26.5,29.0,34.0,12.4440,5.1340
...,...,...,...,...,...,...,...
154,4,12.2,11.5,12.2,13.4,2.0904,1.3936
155,4,13.4,11.7,12.4,13.5,2.4300,1.2690
156,4,12.2,12.1,13.0,13.8,2.2770,1.2558
157,4,19.7,13.2,14.3,15.2,2.8728,2.0672


Separamos los datos Target y Data

In [28]:
y = np.array(df["Peso"]) #sacamos el objetivo
X = np.array(df.drop("Peso", axis=1)) #sacamos el resto de variables eliminando la columna objetivo

Creación de modelo de regresión lineal para entrenamiento de datos.

In [29]:
modelo = LinearRegression() #Creamos el modelo
modelo.fit(X,y) #entrenamos el modelo (todos los datos son de train)

LinearRegression()

In [30]:
modelo.score(X,y)#score solo con train

0.8870971798313481

PRIMERA APROXIMACIÓN. TRAIN_TEST_SPLIT JUNTO CON LINEAR REGRESSION.

Hacemos la división de datos entre train y test usando train_test_split.

In [31]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25) #dividir datos en entrenamiento y test
modelo.fit(X_train,y_train)

LinearRegression()

In [32]:
print("Score del modelo en el conjunto de entrenamiento:", modelo.score(X_train, y_train))
print("Score del modelo en el conjunto de prueba:", modelo.score(X_test, y_test))

Score del modelo en el conjunto de entrenamiento: 0.888729059639728
Score del modelo en el conjunto de prueba: 0.8785906396265575


Podemos observar que el modelo ha obtenido un buen resultado, cercano al 90% de precisión. Por tanto seguiremos probando con otros modelos.

---------------------------------------

EVALUACIÓN CON K-FOLD

In [33]:
# Creamos el kfold
kf = KFold(n_splits=5, shuffle=True)

#Hacemos la validación cruzada con ese kfold
cv_kf = cross_validate(modelo, X, y, cv=kf, return_train_score=True)
cv_kf

{'fit_time': array([0.00156593, 0.00095963, 0.00177598, 0.00131059, 0.00104642]),
 'score_time': array([0.00091982, 0.00063491, 0.00117826, 0.00140405, 0.00076604]),
 'test_score': array([0.89920826, 0.91530916, 0.75304775, 0.85611608, 0.87220252]),
 'train_score': array([0.88164792, 0.87683413, 0.9065821 , 0.89198329, 0.88947351])}

In [34]:
print(cv_kf["train_score"].mean()) # Media de la puntuación de entrenamiento
print(cv_kf["test_score"].mean()) # Media de la puntuación de validación

0.8893041910000645
0.8591767520525305


Como podemos observar, tanto el score de train y test se mantienen cercano al 90%.

------------------------------------

EVALUACIÓN CON SHUFFLESPLIT

In [35]:
# Creamos el ShuffleSplit
ss = ShuffleSplit(n_splits=5, test_size=0.3)

# Realizamos la validación cruzada con ss
cv_ss = cross_validate(modelo, X, y, cv=ss, return_train_score=True)
cv_ss

{'fit_time': array([0.00108576, 0.00111961, 0.00177717, 0.00129628, 0.00117922]),
 'score_time': array([0.00137472, 0.00071716, 0.00130367, 0.0010767 , 0.00072598]),
 'test_score': array([0.80674807, 0.88560672, 0.89061215, 0.89361811, 0.81444434]),
 'train_score': array([0.89829175, 0.88406224, 0.87563398, 0.88130167, 0.89839197])}

In [36]:
print(cv_ss["train_score"].mean()) # Media de la puntuación de entrenamiento
print(cv_ss["test_score"].mean()) # Media de la puntuación de validación

0.8875363225630306
0.858205877830956


Los resultados obtenidos de la validación cruzada con ShuffleSplit son, al igual que con KFold, bastante cercanos al 90% de acierto.

-----------------------------------------------------------

MODELOS DE REGRESIÓN POLINÓMICA

A continuación, realizaremos otro modelo de regresión polinómica. En estos casos, probaremos con grado 2 y 3.

In [37]:
# Prueba con grado 2
pf_2 = PolynomialFeatures(degree=2)
X_pol_2 = pf_2.fit_transform(X)

# Creamos el modelo de linear regression
modelo_pol = LinearRegression()
modelo_pol.fit(X_pol_2, y)

# Creamos un kfold con shuffle
kf_pol_2 = KFold(n_splits=5, shuffle=True)

# Hacemos la validación cruzada con ese kfold
cv_kf_pol_2 = cross_validate(modelo_pol, X_pol_2, y, cv=kf_pol_2, return_train_score=True)


print("Train score con grado 2: ", cv_kf_pol_2["train_score"].mean())
print("Test score con grado 2: ", cv_kf_pol_2["test_score"].mean())


Train score con grado 2:  0.9863633291913153
Test score con grado 2:  0.9651820285319997


Con grado dos hemos obtenido unos resultados muy buenos, ambos en torno al 98% para train score y 96% para test score.
Esto sugiere que este modelo es capaz de generalizar bien.

In [38]:
# Prueba con grado 3
pf_3 = PolynomialFeatures(degree=3)
X_pol_3 = pf_3.fit_transform(X)

# Creamos el modelo de linear regression
modelo_pol = LinearRegression()
modelo_pol.fit(X_pol_3, y)

# Creamos un kfold con shuffle
kf_pol_3 = KFold(n_splits=5, shuffle=True)

# Hacemos la validación cruzada con ese kfold
cv_kf_pol_3 = cross_validate(modelo_pol, X_pol_3, y, cv=kf_pol_3, return_train_score=True)


print("Train score con grado 3: ", cv_kf_pol_3["train_score"].mean())
print("Test score con grado 3: ", cv_kf_pol_3["test_score"].mean())

Train score con grado 3:  0.9959314743703779
Test score con grado 3:  0.8528662411964685


También obtenemos muy buenos resultados con un porcentaje de train score del 99% y un test score del 90%. Pese a tener un test algo más bajo que con un polinomio de grado 2, el polinomio de grado 3 es un buen candidato a mejor modelo.

In [39]:
# Prueba con grado 4
pf_4 = PolynomialFeatures(degree=4)
X_pol_4 = pf_4.fit_transform(X)

# Creamos el modelo de linear regression
modelo_pol = LinearRegression()
modelo_pol.fit(X_pol_4, y)

# Creamos un kfold con shuffle
kf_pol_4 = KFold(n_splits=5, shuffle=True)

# Hacemos la validación cruzada con ese kfold
cv_kf_pol_4 = cross_validate(modelo_pol, X_pol_4, y, cv=kf_pol_4, return_train_score=True)


print("Train score: ", cv_kf_pol_4["train_score"].mean())
print("Test score: ", cv_kf_pol_4["test_score"].mean())

Train score:  0.9999549614224286
Test score:  -191.28757079171675


En este caso, el polinomio con grado 4 produce un sobreajuste, haciendo que el modelo "memorice" los datos de entrenamiento y no generalice bien para datos de prueba, obteniendo un train score de casi 100% pero un pésimo resultado en el test score.

----------------------------------------------------------------

EVALUACIÓN CON MODELO SVR RBF

Vamos a realizar varias pruebas con un modelo svr cuyo kernel sea RBF para encontrar los mejores parámetros.

In [40]:
mejor_score = -np.inf #Inicializamos el mejor score en el infinito negativo
mejor_c = 0
mejor_epsilon = 0
mejor_gamma = 0
c_valores = [283, 285, 284]
ep_valores = [1.6, 1.55, 1.65]
gamma_valores = [0, 0.05, 0.1, 1]
kf_svr = KFold(n_splits=5, shuffle=True)

for c in c_valores:
    for e in ep_valores:
        for g in gamma_valores:
            modelo_svr = SVR(kernel="rbf", C=c, gamma=g, epsilon=e)
            algo = cross_validate(modelo_svr, X, y, cv=kf_svr, return_train_score=True)
            
            if algo["test_score"].mean() > mejor_score:
                mejor_score = algo["test_score"].mean()
                mejor_c = c
                mejor_epsilon = e
                mejor_gamma = g

print("Mejor score:", mejor_score)
print("Mejor C:", mejor_c)
print("Mejor epsilon:", mejor_epsilon)
print("Mejor gamma:", mejor_gamma)


Mejor score: 0.8386022478017698
Mejor C: 284
Mejor epsilon: 1.65
Mejor gamma: 0.05


Una vez obtenidos los mejores hiperparámetros, vamos a volver a crear un modelo usando los hiperparámetros obtenidos anteriormente y vamos a comprobar su score.

In [41]:
modelo_svr = SVR(kernel='rbf', C = mejor_c, epsilon = mejor_epsilon, gamma = mejor_gamma) #Creación del modelo
kf_svr = KFold(n_splits=5, shuffle=True)#Creamos un kfold con shuffle para que "mezcle" los datos

cv_svr = cross_validate(modelo_svr, X, y, cv=kf_svr, return_train_score=True)

print("Train score:", cv_svr["train_score"].mean())
print("Test score:", cv_svr["test_score"].mean())


Train score: 0.9170187844986607
Test score: 0.8255175970760718


Obtenemos un Train Score del 92%, mientras que el Test Score es del 79%. Seguiremos probando otros modelos para ver si obtenemos mejores resultados.

Hacer lo mismo pero con otros modelos de svr y luego grid

-------------------

MODELO SVR LINEAR

Vamos a realizar las pruebas con los mismos hiperparametros que hemos usado en el modelo rbf.

In [42]:
mejor_score = -np.inf #Inicializamos el mejor score en el infinito negativo
mejor_c = 0
mejor_epsilon = 0
mejor_gamma = 0
c_valores = [283, 285, 284]
ep_valores = [1.6, 1.55, 1.65]
gamma_valores = [0, 0.05, 0.1, 1]
kf_svr = KFold(n_splits=5, shuffle=True)


for c in c_valores:
    for e in ep_valores:
        for g in gamma_valores:
            modelo_svr = SVR(kernel="linear", C=c, gamma=g, epsilon=e)
            algo = cross_validate(modelo_svr, X, y, cv=kf_svr, return_train_score=True)
            
            if algo["test_score"].mean() > mejor_score:
                mejor_score = algo["test_score"].mean()
                mejor_c = c
                mejor_epsilon = e
                mejor_gamma = g

print("Mejor score:", mejor_score)
print("Mejor C:", mejor_c)
print("Mejor epsilon:", mejor_epsilon)
print("Mejor gamma:", mejor_gamma)

Mejor score: 0.8606854463365907
Mejor C: 283
Mejor epsilon: 1.55
Mejor gamma: 1


Obtenemos un Score del 86%.

In [43]:
modelo_svr = SVR(kernel='linear', C = mejor_c, epsilon = mejor_epsilon, gamma = mejor_gamma) #Creación del modelo
kf_svr = KFold(n_splits=5, shuffle=True)

cv_svr = cross_validate(modelo_svr, X, y, cv=kf_svr, return_train_score=True)

print("Train score:", cv_svr["train_score"].mean())
print("Test score:", cv_svr["test_score"].mean())

Train score: 0.8677408291074201
Test score: 0.8377338551264188


Obtenemos un train score del 86% y un test score del 84%. Es un modelo bastante aceptable, pero anteriormente hemos obtenido mejores resultados con otros modelos.

------------------------------

MODELO SVR POLINOMIAL

Haremos distintas pruebas con grado 2 y 3. También debemos cambiar el valor de gamma a "scale" ya que de no ser así, da fallos de bucle infinito.

In [44]:
mejor_score = -np.inf #Inicializamos el mejor score en el infinito negativo
mejor_c = 0
mejor_epsilon = 0
mejor_gamma = 0
c_valores = [1, 5, 10]
ep_valores = [1.6, 1.55, 1.65]
#gamma_valores = [0, 0.05, 0.1, 1]
kf_svr = KFold(n_splits=5, shuffle=True)


for c in c_valores:
    for e in ep_valores:
        #for g in gamma_valores:
        modelo_svr = SVR(kernel="poly", C=c, gamma="scale", epsilon=e, degree=2)
        algo = cross_validate(modelo_svr, X, y, cv=kf_svr, return_train_score=True)
        
        if algo["test_score"].mean() > mejor_score:
            mejor_score = algo["test_score"].mean()
            mejor_c = c
            mejor_epsilon = e
            #mejor_gamma = g

print("Mejor score:", mejor_score)
print("Mejor C:", mejor_c)
print("Mejor epsilon:", mejor_epsilon)
print("Mejor gamma:", mejor_gamma)

Mejor score: 0.9083027992028103
Mejor C: 10
Mejor epsilon: 1.55
Mejor gamma: 0


In [45]:
modelo_svr = SVR(kernel='poly', C = mejor_c, epsilon = mejor_epsilon, gamma = "scale", degree = 2) #Creación del modelo
kf_svr = KFold(n_splits=5, shuffle=True)

cv_svr = cross_validate(modelo_svr, X, y, cv=kf_svr, return_train_score=True)

print("Train score:", cv_svr["train_score"].mean())
print("Test score:", cv_svr["test_score"].mean())

Train score: 0.9123309979468261
Test score: 0.9025419786784642


Obtenemos muy buenos valores, ambos scores alrededor del 90% de acierto. 

A continuación, probaremos con grado 3.

In [46]:
mejor_score = -np.inf #Inicializamos el mejor score en el infinito negativo
mejor_c = 0
mejor_epsilon = 0
mejor_gamma = 0
c_valores = [1, 5, 10]
ep_valores = [1.6, 1.55, 1.65]
#gamma_valores = [0, 0.05, 0.1, 1]
kf_svr = KFold(n_splits=5, shuffle=True)


for c in c_valores:
    for e in ep_valores:
        #for g in gamma_valores:
        modelo_svr = SVR(kernel="poly", C=c, gamma="scale", epsilon=e, degree=3)
        algo = cross_validate(modelo_svr, X, y, cv=kf_svr, return_train_score=True)
        
        if algo["test_score"].mean() > mejor_score:
            mejor_score = algo["test_score"].mean()
            mejor_c = c
            mejor_epsilon = e
            #mejor_gamma = g

print("Mejor score:", mejor_score)
print("Mejor C:", mejor_c)
print("Mejor epsilon:", mejor_epsilon)
print("Mejor gamma:", mejor_gamma)

Mejor score: 0.9441724975817168
Mejor C: 10
Mejor epsilon: 1.65
Mejor gamma: 0


In [47]:
modelo_svr = SVR(kernel='poly', C = mejor_c, epsilon = mejor_epsilon, gamma = "scale", degree = 3) #Creación del modelo
kf_svr = KFold(n_splits=5, shuffle=True)

cv_svr = cross_validate(modelo_svr, X, y, cv=kf_svr, return_train_score=True)

print("Train score:", cv_svr["train_score"].mean())
print("Test score:", cv_svr["test_score"].mean())

Train score: 0.9481249490416218
Test score: 0.9434844690721376


Obtenemos un scores algo más elevados que con el modelo de grado 2.

----------------------------------

MODELO SVR SIGMOIDE

En esta ocasión, vamos a seguir usando los mismos hiperparámetros que en los ejercicios anteriores.

In [48]:
mejor_score = -np.inf #Inicializamos el mejor score en el infinito negativo
mejor_c = 0
mejor_epsilon = 0
mejor_gamma = 0
c_valores = [283, 285, 284]
ep_valores = [1.6, 1.55, 1.65]
gamma_valores = [0, 0.05, 0.1, 1]
kf_svr = KFold(n_splits=5, shuffle=True)


for c in c_valores:
    for e in ep_valores:
        for g in gamma_valores:
            modelo_svr = SVR(kernel="sigmoid", C=c, gamma=g, epsilon=e)
            algo = cross_validate(modelo_svr, X, y, cv=kf_svr, return_train_score=True)
            
            if algo["test_score"].mean() > mejor_score:
                mejor_score = algo["test_score"].mean()
                mejor_c = c
                mejor_epsilon = e
                mejor_gamma = g

print("Mejor score:", mejor_score)
print("Mejor C:", mejor_c)
print("Mejor epsilon:", mejor_epsilon)
print("Mejor gamma:", mejor_gamma)

Mejor score: -0.10970911032897371
Mejor C: 283
Mejor epsilon: 1.55
Mejor gamma: 1


In [49]:
modelo_svr = SVR(kernel='sigmoid', C = mejor_c, epsilon = mejor_epsilon, gamma = "scale") #Creación del modelo
kf_svr = KFold(n_splits=5, shuffle=True)

cv_svr = cross_validate(modelo_svr, X, y, cv=kf_svr, return_train_score=True)

print("Train score:", cv_svr["train_score"].mean())
print("Test score:", cv_svr["test_score"].mean())

Train score: -19.740062204044293
Test score: -20.560562886948013


Como podemos observar, obtenemos un valores malísimos, entorno al -20.

------------------------------------------------
------------------------------------------------

MODELO USANDO GRIDSEARCH

A continuación usaremos GridSearch para encontrar los mejores hiperparámetros para el modelo SVR polinomial de grado 3, ya que es el que mejor resultado dio anteriormente.

Para ello, cusaremos las variables X_train, y_train, X_test y y_test que ya se han creado anteriormente.

Lo siguiente será definir un diccionario con los hiperparámetros.

In [50]:
#Definimos un diccionario con los hiperparametros a probar
hiperparametros = {
    'C': [1, 10, 50, 100,200, 300, 400, 60, 70, 80, 500],
    'gamma': ["scale"],
    'epsilon': [0.1, 0.01, 1 ,0.001, 0.0001, 2, 3, 10, 30, 50, 60, 70, 80]
}

model = SVR(kernel="poly", degree=3)#Creamos el modelo
gs = GridSearchCV(model, hiperparametros, cv=5)#Creamos un GridSearchCV con el modelo SVR y los hiperparametros
gs.fit(X_train, y_train)#Lo entrenamos

print("Mejor score:", gs.best_score_)
print("Mejor C:", gs.best_params_['C'])
print("Mejor gamma:", gs.best_params_['gamma'])
print("Mejor epsilon:", gs.best_params_['epsilon'])
print("Score del test:", gs.best_estimator_.score(X_test, y_test))



Mejor score: 0.9563165518465974
Mejor C: 60
Mejor gamma: scale
Mejor epsilon: 0.0001
Score del test: 0.9556993773497193


Obtenemos un gran resultado con el GridSearchCV para un polinomio de grado 3.

Como conclusión, demostramos que el mejor modelo para predecir el peso de los peces es el modelo SVR con kernel Poly y grado 3 al cual le hemos realizado un GridSearchCV para optimizar los hiperparámetros.